In [2]:
import langchain

In [3]:
### intoduction to data ingestion

In [4]:
import os
from typing import List, Dict, Any
import pandas as pd


In [3]:
from langchain_core.documents import Document
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
)
print("setup complete")

setup complete


In [8]:
## Understading the document structure

In [4]:
doc = Document(
    page_content="This is a sample document.",
    metadata={"source": "S3", "author": "Ashish Agnihotri", "date": "2024-06-15", "tags": ["sample", "document"], "version": 1.0}
)

print("Document content:", doc.page_content)
print("Document metadata:", doc.metadata)

Document content: This is a sample document.
Document metadata: {'source': 'S3', 'author': 'Ashish Agnihotri', 'date': '2024-06-15', 'tags': ['sample', 'document'], 'version': 1.0}


TEXT FILE READING 

In [1]:
import os
os.makedirs("data/textfiles", exist_ok=True)

In [2]:
sample_text = {
    "data/textfiles/sample2.txt": """Python is a high-level, general-purpose programming language created by Guido van Rossum and first released in 1991. It's known for readable code that looks close to plain English, which makes it one of the most popular first languages and one of the most widely used languages overall.

Key features

Simple syntax: Indentation defines code blocks, so there are no curly braces or semicolons.
Interpreted: You run code directly without a separate compile step, which makes testing quick.
Dynamically typed: You don't declare variable types, though optional type hints are available.
Multi-paradigm: It supports procedural, object-oriented and functional styles.
Huge ecosystem: Hundreds of thousands of packages are available through pip and PyPI.
Cross-platform: It runs on Windows, macOS and Linux.

What Python is used for

Data science and analytics: pandas, NumPy, Matplotlib
Machine learning and AI: PyTorch, TensorFlow, scikit-learn (this is where RAG and LLM apps live too)
Web development: Django, Flask, FastAPI
Automation and scripting: file handling, web scraping, bots
Testing, DevOps and cloud tooling
Education: it's a common first language in schools and bootcamps".
"""}

In [3]:
print(sample_text)

{'data/textfiles/sample2.txt': 'Python is a high-level, general-purpose programming language created by Guido van Rossum and first released in 1991. It\'s known for readable code that looks close to plain English, which makes it one of the most popular first languages and one of the most widely used languages overall.\n\nKey features\n\nSimple syntax: Indentation defines code blocks, so there are no curly braces or semicolons.\nInterpreted: You run code directly without a separate compile step, which makes testing quick.\nDynamically typed: You don\'t declare variable types, though optional type hints are available.\nMulti-paradigm: It supports procedural, object-oriented and functional styles.\nHuge ecosystem: Hundreds of thousands of packages are available through pip and PyPI.\nCross-platform: It runs on Windows, macOS and Linux.\n\nWhat Python is used for\n\nData science and analytics: pandas, NumPy, Matplotlib\nMachine learning and AI: PyTorch, TensorFlow, scikit-learn (this is wh

In [5]:
for file_path, content in sample_text.items():
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)
print("data ingestion complete")

data ingestion complete


In [7]:

from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/textfiles/sample2.txt")

documents = loader.load()
print(f"Loaded {len(documents)} documents.")
print("First document content:", documents[0].page_content[:100])
print("First document metadata:", documents[0].metadata)
print("Data ingestion and loading complete.")

Loaded 1 documents.
First document content: Python is a high-level, general-purpose programming language created by Guido van Rossum and first r
First document metadata: {'source': 'data/textfiles/sample2.txt'}
Data ingestion and loading complete.


In [ ]:
### directory loader  - mulptiple files

In [11]:
from langchain_community.document_loaders import DirectoryLoader

directory_loader = DirectoryLoader("data/textfiles", glob="*.txt")

dir_loader= DirectoryLoader("data/textfiles", glob="*.txt",
                            loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True
)

documents = dir_loader.load()

for doc in documents:
    print(f"Document content: {doc.page_content[:100]}")
    print(f"Document metadata: {doc.metadata}")
print(f"Loaded {len(documents)} documents from directory.")
print("First document content:", documents[0].page_content[:100])
print("First document metadata:", documents[0].metadata)
print("Directory loading complete.")

100%|██████████| 3/3 [00:00<00:00, 607.08it/s]

Document content: RANDOM INFORMATION FILE 1: SCIENCE AND NATURE

1. Hon
Document metadata: {'source': 'data\\textfiles\\random_info_1_science_nature.txt'}
Document content: RANDOM INFORMATION FILE 2: HISTORY, TECH, LANGUAGE AND THE WORLD
Document metadata: {'source': 'data\\textfiles\\random_info_2_history_tech_world.txt'}
Document content: Python is a high-level, general-purpose programming language created by Guido van Rossum and first r
Document metadata: {'source': 'data\\textfiles\\sample2.txt'}
Loaded 3 documents from directory.
First document content: RANDOM INFORMATION FILE 1: SCIENCE AND NATURE

1. Hon
First document metadata: {'source': 'data\\textfiles\\random_info_1_science_nature.txt'}
Directory loading complete.


In [1]:
##text splitter 

In [2]:
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
    MarkdownHeaderTextSplitter,
    HTMLHeaderTextSplitter,
    RecursiveJsonSplitter,
    Language,
    SentenceTransformersTokenTextSplitter,
)

In [3]:
SAMPLE_TEXT = """LangChain is a framework for building applications with large language models.
 
It provides abstractions for prompts, chains, agents, memory, and retrieval.
 
Retrieval-augmented generation (RAG) needs documents split into smaller chunks. 
Good chunking improves retrieval quality and keeps prompts within the context window.
Chunk size and overlap are the two most important parameters to tune."""
 
 
def show(title, chunks):
    print(f"\n===== {title} =====")
    for i, c in enumerate(chunks, 1):
        text = c.page_content if hasattr(c, "page_content") else c
        print(f"[{i}] ({len(text)} chars) {text!r}")

In [9]:
# 1. CharacterTextSplitter  (splits on ONE separator)
# ---------------------------------------------------------------
def character_splitter():
    splitter = CharacterTextSplitter(
        separator="\n\n",      # split on paragraphs
        chunk_size=150,
        chunk_overlap=20,
        length_function=len,
    )
    show("CharacterTextSplitter", splitter.split_text(SAMPLE_TEXT))
    return splitter.split_text(SAMPLE_TEXT)

character_splitter()




===== CharacterTextSplitter =====
[1] (396 chars) 'LangChain is a framework for building applications with large language models.\n \nIt provides abstractions for prompts, chains, agents, memory, and retrieval.\n \nRetrieval-augmented generation (RAG) needs documents split into smaller chunks. \nGood chunking improves retrieval quality and keeps prompts within the context window.\nChunk size and overlap are the two most important parameters to tune.'


['LangChain is a framework for building applications with large language models.\n \nIt provides abstractions for prompts, chains, agents, memory, and retrieval.\n \nRetrieval-augmented generation (RAG) needs documents split into smaller chunks. \nGood chunking improves retrieval quality and keeps prompts within the context window.\nChunk size and overlap are the two most important parameters to tune.']

In [8]:
def recursive_splitter():
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=150,
        chunk_overlap=30,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    show("RecursiveCharacterTextSplitter", splitter.split_text(SAMPLE_TEXT))

recursive_splitter()


===== RecursiveCharacterTextSplitter =====
[1] (78 chars) 'LangChain is a framework for building applications with large language models.'
[2] (76 chars) 'It provides abstractions for prompts, chains, agents, memory, and retrieval.'
[3] (79 chars) 'Retrieval-augmented generation (RAG) needs documents split into smaller chunks.'
[4] (85 chars) 'Good chunking improves retrieval quality and keeps prompts within the context window.'
[5] (69 chars) 'Chunk size and overlap are the two most important parameters to tune.'


In [ ]:
# 3. Token-based splitting (respect LLM token limits)
# ---------------------------------------------------------------
def token_splitter():
    # a) Pure token splitter (uses tiktoken)
    splitter = TokenTextSplitter(
        encoding_name="cl100k_base",
        chunk_size=40,
        chunk_overlap=5,
    )
    show("TokenTextSplitter", splitter.split_text(SAMPLE_TEXT))
 
    # b) Recursive splitter, but length measured in tokens (best of both)
    splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        encoding_name="cl100k_base",
        chunk_size=40,
        chunk_overlap=5,
    )
    show("Recursive + tiktoken", splitter.split_text(SAMPLE_TEXT))
 
    # c) Sentence-transformers tokenizer (matches embedding model limits)
    splitter = SentenceTransformersTokenTextSplitter(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        chunk_overlap=0,
        tokens_per_chunk=32,
    )
    show("SentenceTransformersTokenTextSplitter", splitter.split_text(SAMPLE_TEXT))
    
token_splitter()

In [12]:
def markdown_splitter():
    md = """# Guide
## Install
Run pip install langchain.
## Usage
### Basic
Import and call the chain.
### Advanced
Use agents and tools.
"""
    print("\n" + "=" * 60 + "\nMarkdownHeaderTextSplitter\n" + "=" * 60)
    print("INPUT:\n" + md)
    headers = [("#", "h1"), ("##", "h2"), ("###", "h3")]
    md_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers,
        strip_headers=False,
    )
    docs = md_splitter.split_text(md)
    for d in docs:
        print("--- chunk ---")
        print("metadata:", d.metadata)
        print("content :", d.page_content)
 
    # Common pattern: header split first, then size-limit the pieces
    size_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)
    final_docs = size_splitter.split_documents(docs)
    print(f"\nMarkdown -> {len(final_docs)} final chunks")

markdown_splitter()


MarkdownHeaderTextSplitter
INPUT:
# Guide
## Install
Run pip install langchain.
## Usage
### Basic
Import and call the chain.
### Advanced
Use agents and tools.

--- chunk ---
metadata: {'h1': 'Guide', 'h2': 'Install'}
content : # Guide  
## Install
Run pip install langchain.
--- chunk ---
metadata: {'h1': 'Guide', 'h2': 'Usage', 'h3': 'Basic'}
content : ## Usage  
### Basic
Import and call the chain.
--- chunk ---
metadata: {'h1': 'Guide', 'h2': 'Usage', 'h3': 'Advanced'}
content : ### Advanced
Use agents and tools.

Markdown -> 3 final chunks


In [15]:
def markdown_splitter():
    md = """# Guide
## Install
Run pip install langchain.
## Usage
### Basic
Import and call the chain.
### Advanced
Use agents and tools.
"""
    print("\n" + "=" * 60 + "\nMarkdownHeaderTextSplitter\n" + "=" * 60)
    print("INPUT:\n" + md)
    headers = [("#", "h1"), ("##", "h2"), ("###", "h3")]
    md_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers,
        strip_headers=False,
    )
    docs = md_splitter.split_text(md)
    for d in docs:
        print("--- chunk ---")
        print("metadata:", d.metadata)
        print("content :", d.page_content)
 
    # Common pattern: header split first, then size-limit the pieces
    size_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)
    final_docs = size_splitter.split_documents(docs)
    print(f"\nMarkdown -> {len(final_docs)} final chunks")

markdown_splitter()


MarkdownHeaderTextSplitter
INPUT:
# Guide
## Install
Run pip install langchain.
## Usage
### Basic
Import and call the chain.
### Advanced
Use agents and tools.

--- chunk ---
metadata: {'h1': 'Guide', 'h2': 'Install'}
content : # Guide  
## Install
Run pip install langchain.
--- chunk ---
metadata: {'h1': 'Guide', 'h2': 'Usage', 'h3': 'Basic'}
content : ## Usage  
### Basic
Import and call the chain.
--- chunk ---
metadata: {'h1': 'Guide', 'h2': 'Usage', 'h3': 'Advanced'}
content : ### Advanced
Use agents and tools.

Markdown -> 3 final chunks


In [21]:
import sys
import pymupdf  # PyMuPDF (older code uses: import fitz)
from langchain_community.document_loaders import PyPDFLoader
 
PDF_PATH = "C:\\Users\\ashis\\OneDrive\\Desktop\\RAG_udemy\\0-DataParsing\\data\\pdf\\sample_rag_guide.pdf"
 
# ---------------------------------------------------------------
# 1. PyMuPDF: direct, fast, low-level access
# ---------------------------------------------------------------
print("=" * 70)
print("1. PyMuPDF")
print("=" * 70)
 
with pymupdf.open(PDF_PATH) as doc:
    print("Pages   :", doc.page_count)
    print("Metadata:", doc.metadata)
 
    for page_no, page in enumerate(doc, start=1):
        text = page.get_text("text")          # plain text of the page
        print(f"\n--- Page {page_no} ({len(text)} chars) ---")
        print(text)

ImportError: DLL load failed while importing _extra: The specified module could not be found.